In [1]:
""" LangExtract Product Information Extraction Example"""


from concurrent.futures import ThreadPoolExecutor
from Modules.hyperparameters import get_hyperparameters
from Modules.load_db_cleaning import join_datasets
from Modules.load_db import get_dataframe_dict
import json
import langextract as lx
import textwrap
import pandas as pd

In [2]:
""" Constants """

DATASET = 'jsonllm'
HYPER = get_hyperparameters()
# Set to 0 to process all entries
STARTING_POINT = HYPER[DATASET]['starting_point']
# Number of threads for parallel processing
MAX_WORKERS = HYPER[DATASET]['workers']
# Output file path
OUTPUT_PATH = HYPER[DATASET]['output_path']

PROMPT = textwrap.dedent(HYPER['prompts']['system']['mave_task'])

In [3]:
""" Functions: get_lex_Example, get_examples, check_last_jsonl_id """


def get_lex_Example(example_item: dict) -> lx.data.ExampleData:
    return lx.data.ExampleData(
        text=example_item['text'],
        extractions=[
            lx.data.Extraction(
                extraction_class=example_item['class'],
                extraction_text=example_item['ext_text'],
                attributes=example_item['attributes'],
            ),
        ],
    )


def get_examples() -> list[lx.data.ExampleData]:
    """ Generate example data for LangExtract. """

    examples_data = [
        {'id': 1,
         'text': 'Camiseta PoloTech masculina de algodão, cor azul marinho, disponível nos tamanhos M, G e GG.',
         'class': 'product',
         'ext_text': 'Camiseta PoloTech masculina',
            'attributes': {
                'brand': 'PoloTech',
                'category': 'camiseta',
                'material': 'algodão',
                'color': 'azul marinho',
                'sizes': ['M', 'G', 'GG']
            }
         },
        {'id': 2,
         'text': 'Tênis esportivo Nike Air Zoom branco, ideal para corrida.',
         'class': 'product',
         'ext_text': 'Tênis esportivo Nike Air Zoom branco',
            'attributes': {
                'brand': 'Nike',
                'category': 'tênis esportivo',
                'color': 'branco',
                'intended_use': 'corrida'
            }
         },
    ]
    examples = []
    for example_item in examples_data:
        examples.append(get_lex_Example(example_item))
    return examples


def check_last_jsonl_id() -> int:
    """ Check the last processed ID in the output file to resume processing. """
    try:
        with open(OUTPUT_PATH, "r") as f:
            lines = f.readlines()
            if lines:
                last_record = json.loads(lines[-2])
                starting_point = last_record.get("id", -1) + 1
                if HYPER['mave']['verbose']:
                    print(f"Resuming from ID: {starting_point}")
                return starting_point
    except FileNotFoundError:
        return STARTING_POINT
    return STARTING_POINT

In [ ]:
""" Second Round of Constants """

STARTING_POINT = check_last_jsonl_id()

datasets_dict = get_dataframe_dict(['ae-110k', 'oa-mine', 'mave'])
JSONLLM = join_datasets(datasets_dict)
SAFE_TEXTS = JSONLLM['text'][STARTING_POINT:].tolist()

Loading dataframe: ae-110k
Loading dataframes from local path: b:\GitHub\UFMG\JSONLLM\Files\Code\Datasets\ae-110k
General Clean: drop_na, keep bigger json_answer, re-index
Shape Change: from (39505, 13) to (39505, 13)
Splitting dataframe into train, validation and test sets with proportions: {'test': 0.1, 'validation': 0.1, 'train': 0.8}
Loaded dataframes with splits: ['test', 'validation', 'train']
The shapes of the dataframes are: {'test': (3950, 13), 'validation': (3950, 13), 'train': (31605, 13)}
Loading dataframe: oa-mine
Loading dataframes from local path: b:\GitHub\UFMG\JSONLLM\Files\Code\Datasets\oa-mine
General Clean: drop_na, keep bigger json_answer, re-index
Shape Change: from (1943, 11) to (1943, 11)
Splitting dataframe into train, validation and test sets with proportions: {'test': 0.1, 'validation': 0.1, 'train': 0.8}
Loaded dataframes with splits: ['test', 'validation', 'train']
The shapes of the dataframes are: {'test': (194, 11), 'validation': (194, 11), 'train': (1555

In [5]:
""" More Functions """


def save_record_to_file(idx: int, record: dict, file) -> None:
    """ Save a single record to the output file. """
    file.write(json.dumps(record, ensure_ascii=False) + "\n")
    if record and "error" not in record:
        print(f'[{idx}] OK')
    else:
        print(f'[{idx}] ERROR: {record}')


def format_output(idx: int, row: pd.Series, extraction_result: dict) -> dict:
    """ Format the extraction result into the desired output structure. """
    text = row.get('text', None)
    # Montar campos no mesmo formato do AE-110K
    attrs = extraction_result.extractions[0].attributes or {}
    attributes = list(attrs.keys())
    values = list(attrs.values())
    values_indices = []
    values_text = " | ".join(map(str, values))
    attributes_values = " | ".join(
        f"attribute: {k}, value: {v}" for k, v in attrs.items()
    )
    # json_answer = str(attrs)  # igual ao df (aspas simples)
    json_answer = json.dumps(attrs, ensure_ascii=False)

    record = {
        "id": row.get('id', None),
        'dataset': DATASET,
        'split': row.get('split', None),
        'text': text,
        'json_answer': json_answer,
        'source': row.get('source', None),
        'category': row.get('categories', None),
        'attributes': attributes,
        'attributes_values': attributes_values,
        'values': values,

        'values_indices': None,
        'candidate_example': None,
        'candidate_text': None,
    }
    return record


def extract_text(idx: int, row: pd.Series) -> tuple[int, dict | None]:
    # print(f'Row: {row.get('text', None)}')
    text = row.get('text', None)
    try:
        result = lx.extract(
            text_or_documents=text,
            prompt_description=PROMPT,
            examples=get_examples(),
            model_id=HYPER[DATASET]['model_id'],
            model_url=HYPER[DATASET]['model_url'],
            fence_output=False,
            # max_workers=10,
            use_schema_constraints=False,
            language_model_params={"timeout": 900}
        )

        if not result.extractions:
            return idx, None
        record = format_output(idx, row, result)
        return idx, record
    except Exception as err:
        return idx, {"error": str(err)}

In [ ]:
""" Testing """


def testing():
    # iterate rows as (idx, Series) so extract_text can use row.get(...) and row['text']
    rows = JSONLLM.iloc[0:9].copy()
    with open(OUTPUT_PATH, "a", encoding="utf-8") as file:
        for idx, row in rows.iterrows():
            print(row.text)
            # idx, record = extract_text(idx, row)
            # save_record_to_file(idx, record, file)


testing()

In [ ]:
""" Paralelização """


def parallel_extraction(workers: int = MAX_WORKERS, output_path: str = OUTPUT_PATH):
    """ Perform parallel extraction and save results to output file. """
    print(f'Saving results to {output_path} using {workers} workers.')
    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor, open(output_path, "a") as f:
        for i, record in executor.map(lambda args: extract_text(*args), JSONLLM[STARTING_POINT:].iterrows()):
            # for i, record in executor.map(lambda args: extract_text(*args), enumerate(SAFE_TEXTS, start=STARTING_POINT)):
            save_record_to_file(i, record, f)

In [ ]:
parallel_extraction()